# Prompt Caching with Amazon Bedrock (Run in windows)

**What is prompt caching?**

Normally, every API call re-processes your entire prompt from scratch — even if most of it never changes (e.g. a large system prompt, a long document, or a reference corpus). This wastes time and money.

**Prompt caching** lets Bedrock save ('cache') the processed state of a prompt prefix. Subsequent calls that reuse that same prefix skip the reprocessing entirely — they get a **cache hit**, which is significantly faster and cheaper.

**This demo shows:**
1. A baseline call with no caching
2. A cache WRITE call (first use — Bedrock processes and saves the prefix)
3. A cache READ call (subsequent use — Bedrock retrieves from cache)
4. A side-by-side comparison of latency and token costs



## 1. Setup

In [6]:
import boto3
import json
import time

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

MODEL_ID = "us.anthropic.claude-sonnet-4-6"

# MODEL_ID = "us.anthropic.claude-haiku-5-5"

print(f"Client ready. Using model: {MODEL_ID}")

Client ready. Using model: us.anthropic.claude-sonnet-4-6


## 2. Create a large reusable context

Prompt caching is most valuable when you have a large, stable block of text that you send repeatedly — like a product manual, a legal document, or detailed instructions. We'll simulate that here with a long technical reference document.

In [7]:
# A large reference document — in real use this could be a product manual,
# codebase, legal document, or any long stable context
LARGE_CONTEXT = """
Prompt caching for faster model inference
Prompt caching is an optional feature that you can use with supported models on Amazon Bedrock
to reduce inference response latency and input token costs. By adding portions of your context to
a cache, the model can use the cache to skip recomputation of inputs, allowing Bedrock to share in
the compute savings and lower your response latencies.
Prompt caching can help when you have workloads with long and repeated contexts that are
frequently reused for multiple queries. For example, if you have a chatbot where users can upload
documents and ask questions about them, it can be time consuming for the model to process the
document every time the user provides input. With prompt caching, you can cache the document
so that future queries containing the document don't need to reprocess it.
When using prompt caching, you're charged at a reduced rate for tokens read from cache.
Depending on the model, tokens written to cache may be charged at a rate that is higher than
that of uncached input tokens. Any tokens not read from or written to cache, are charged at the
standard input token rate for that model. For more information, see the Amazon Bedrock pricing
page.
How it works
If you opt to use prompt caching, Amazon Bedrock creates a cache composed of cache checkpoints.
These are markers that define the contiguous subsection of your prompt that you wish to cache
(often referred to as a prompt prefix). These prompt prefixes should be static between requests,
alterations to the prompt prefix in subsequent requests will result in cache misses.
Request a quota increase 2317
Amazon Bedrock User Guide
Cache checkpoints have a minimum and maximum number of tokens, dependent on the specific
model you're using. You can only create a cache checkpoint if your total prompt prefix meets the
minimum number of tokens. For example, Claude 3.7 Sonnet requires at least 1,024 tokens per
cache checkpoint, while Claude Opus 4.5, Claude Opus 4.6, Claude Haiku 4.5, and Claude Sonnet
4.5 require at least 4,096 tokens per cache checkpoint. That means that for a model with a 1,024-
token minimum, your first cache checkpoint can be defined after 1,024 tokens and your second
cache checkpoint can be defined after 2,048 tokens. If you try to add a cache checkpoint before
meeting the minimum number of tokens, your inference will still succeed, but your prefix will not
be cached. The cache has a Time To Live (TTL), which resets with each successful cache hit. During
this period, the context in the cache is preserved. If no cache hits occur within the TTL window,
your cache expires. Many models support a 5-minute TTL. Check the model card for your model to
see the exact TTL conditions.
You can use prompt caching anytime you get model inference in Amazon Bedrock for supported
models. Prompt caching is supported by the following Amazon Bedrock features:
Converse and ConverseStream APIs
You can carry on a conversation with a model where you specify cache checkpoints in your
prompts.
InvokeModel and InvokeModelWithResponseStream APIs
You can submit single prompt requests in which you enable prompt caching and specify your
cache checkpoints.
Prompt Caching with Cross-region Inference
Prompt caching can be used in conjunction with cross region inference. Cross-region inference
automatically selects the optimal AWS Region within your geography to serve your inference
request, thereby maximizing available resources and model availability. At times of high
demand, these optimizations may lead to increased cache writes.
Amazon Bedrock Prompt management
When you create or modify a prompt, you can choose to enable prompt caching. Depending
on the model, you can cache system prompts, system instructions, and messages (user and
assistant). You can also choose to disable prompt caching.
How it works 2318
Amazon Bedrock User Guide
Note
Prompt caching is only supported for on-demand inference endpoints. It is not supported
with the batch inference API.
The APIs provide you with the most flexibility and granular control over the prompt cache. You
can set an individual cache checkpoint within your prompts. You can add to the cache by creating
more cache checkpoints, up to the maximum number of cache checkpoints allowed for the specific
model. For more information, see Supported models, Regions, and limits.
Simplified Cache Management for Claude Models
For Claude models, Amazon Bedrock offers a simplified approach to cache management that
reduces the complexity of manually placing cache checkpoints. Instead of requiring you to
specify exact cache checkpoint locations, you can use automatic cache management with a single
breakpoint at the end of your static content.
When you enable simplified cache management, the system automatically checks for cache hits at
previous content block boundaries, looking back up to approximately 20 content blocks from your
specified breakpoint. This allows the model to find the longest matching prefix from your cache
without requiring you to predict the optimal checkpoint locations. To use this, place a single cache
Simplified Cache Management for Claude Models 2321
Amazon Bedrock User Guide
checkpoint at the end of your static content, before any dynamic or variable content. The system
will automatically find the best cache match.
For more granular control, you can still use multiple cache checkpoints (up to 4 for Claude models)
to specify exact cache boundaries. You should use multipled cache checkpoints if you are caching
sections that change at different frequencies or want more control over exactly what gets cached.
Important
The automatic prefix checking only looks back approximately 20 content blocks from
your cache checkpoint. If your static content extends beyond this range, consider using
multiple cache checkpoints or restructuring your prompt to place the most frequently
reused content within this range.
How to effectively use prompt caching
If you have prompts that are used at a regular cadence (i.e., system prompts that are used more
frequently than every 5 minutes), continue to use the 5-minute cache, since this will continue to be
refreshed at no additional charge.
The 1-hour cache is best used in the following scenarios:
• When you have prompts that are likely used less frequently than 5 minutes, but more frequently
than every hour. For example, when an agentic side-agent will take longer than 5 minutes, or
when storing a long chat conversation with a user and you generally expect that user may not
respond in the next 5 minutes.
• When latency is important and your follow-up prompts may be sent beyond 5 minutes.
• When you want to improve your rate limit use, since cache hits are not deducted against your
rate limit.
You can use both 1-hour and 5-minute cache controls in the same request, but with an important
constraint: Cache entries with longer TTL must appear before shorter TTLs (i.e., a 1-hour cache
entry must appear before any 5-minute cache entries).
How to effectively use prompt caching 2322
Amazon Bedrock User Guide
Getting started
The following sections show you a brief overview of how to use the prompt caching feature for
each method of interacting with models through Amazon Bedrock.
Converse API
The Converse API provides advanced and flexible options for implementing prompt caching in
multi-turn conversations. For more information about the prompt requirements for each model,
see the preceding section Supported models, Regions, and limits.
Example request
The following examples show a cache checkpoint set in the messages, system, or tools fields
of a request to the Converse API. You can place checkpoints in any of these locations for a given
request. For example, if sending a request to the Claude 3.5 Sonnet v2 model, you could place two
cache checkpoints in messages, one cache checkpoint in system, and one in tools. For more
detailed information and examples of structuring and sending Converse API requests, see Inference
using Converse API.

"""

word_count = len(LARGE_CONTEXT.split())
print(f"Context document: {word_count} words, {len(LARGE_CONTEXT)} characters")
print("This simulates a large stable reference document you might reuse across many queries.")

Context document: 1294 words, 8035 characters
This simulates a large stable reference document you might reuse across many queries.


## 3. Baseline — No caching

First, we make a call the normal way — no cache point. The full context is processed every time.

In [8]:
def call_without_cache(question):
    """Standard call — no caching, full context processed every time."""
    start = time.time()
    
    response = bedrock.converse(
        modelId=MODEL_ID,
        system=[{
            "text": f"You are a helpful AWS expert. Use the following reference document to answer questions accurately.\n\n{LARGE_CONTEXT}"
        }],
        messages=[{
            "role": "user",
            "content": [{"text": question}]
        }]
    )
    
    elapsed = time.time() - start
    usage = response['usage']
    answer = response['output']['message']['content'][0]['text']
    
    return {
        "answer": answer,
        "elapsed": elapsed,
        "input_tokens": usage.get('inputTokens', 0),
        "output_tokens": usage.get('outputTokens', 0),
        "cache_read_tokens": 0,
        "cache_write_tokens": 0
    }

print("Running baseline call (no cache)...")
baseline = call_without_cache("What is prompt caching and what are its key features?")

print(f"\nAnswer: {baseline['answer'][:200]}...")
print(f"\n⏱  Time:          {baseline['elapsed']:.2f}s")
print(f"📥  Input tokens:  {baseline['input_tokens']}")
print(f"📤  Output tokens: {baseline['output_tokens']}")

Running baseline call (no cache)...

Answer: # Prompt Caching on Amazon Bedrock

## What is Prompt Caching?

Prompt caching is an **optional feature** available on Amazon Bedrock that allows you to cache portions of your prompt context, enabling...

⏱  Time:          7.14s
📥  Input tokens:  1839
📤  Output tokens: 704


## 4. With Caching — Cache WRITE (first call)

Now we add a `cachePoint` marker after the large context in the system prompt. This tells Bedrock: *"cache everything up to this point."*

The **first call** is a cache WRITE — Bedrock processes the prefix and saves it. You'll see `cacheWriteInputTokens` in the usage stats.

In [9]:
def call_with_cache(question):
    """Call with a cache point — Bedrock caches the large context prefix."""
    start = time.time()
    
    response = bedrock.converse(
        modelId=MODEL_ID,
        system=[
            {
                "text": f"You are a helpful AWS expert. Use the following reference document to answer questions accurately.\n\n{LARGE_CONTEXT}"
            },
            {
                # This is the cache point — everything above this gets cached
                "cachePoint": {"type": "default"}
            }
        ],
        messages=[{
            "role": "user",
            "content": [{"text": question}]
        }]
    )
    
    elapsed = time.time() - start
    usage = response['usage']
    answer = response['output']['message']['content'][0]['text']
    
    return {
        "answer": answer,
        "elapsed": elapsed,
        "input_tokens": usage.get('inputTokens', 0),
        "output_tokens": usage.get('outputTokens', 0),
        "cache_read_tokens": usage.get('cacheReadInputTokens', 0),
        "cache_write_tokens": usage.get('cacheWriteInputTokens', 0)
    }

print("Running FIRST cached call (cache WRITE)...")
cache_write = call_with_cache("What is prompt caching and what are its key features?")

print(f"\nAnswer: {cache_write['answer'][:200]}...")
print(f"\n⏱  Time:                {cache_write['elapsed']:.2f}s")
print(f"📥  Input tokens:        {cache_write['input_tokens']}")
print(f"📤  Output tokens:       {cache_write['output_tokens']}")
print(f"✍️  Cache WRITE tokens:  {cache_write['cache_write_tokens']}  ← context written to cache")
print(f"📖  Cache READ tokens:   {cache_write['cache_read_tokens']}")

Running FIRST cached call (cache WRITE)...

Answer: # Prompt Caching on Amazon Bedrock

## What is Prompt Caching?

Prompt caching is an **optional feature** available on Amazon Bedrock that allows you to cache portions of your prompt context so the mo...

⏱  Time:                6.86s
📥  Input tokens:        17
📤  Output tokens:       665
✍️  Cache WRITE tokens:  1822  ← context written to cache
📖  Cache READ tokens:   0


## 5. With Caching — Cache READ (second call, different question)

Now we ask a **different question** using the exact same system prompt with cache point. Bedrock retrieves the cached prefix instead of reprocessing it.

You'll see `cacheReadInputTokens` in the usage — and the call should be noticeably faster.

In [10]:
print("Running SECOND cached call (cache READ, different question)...")
cache_read = call_with_cache("What TTLs are supported for cache entries?")

print(f"\nAnswer: {cache_read['answer'][:200]}...")
print(f"\n⏱  Time:                {cache_read['elapsed']:.2f}s")
print(f"📥  Input tokens:        {cache_read['input_tokens']}")
print(f"📤  Output tokens:       {cache_read['output_tokens']}")
print(f"✍️  Cache WRITE tokens:  {cache_read['cache_write_tokens']}")
print(f"📖  Cache READ tokens:   {cache_read['cache_read_tokens']}  ← context served from cache")

Running SECOND cached call (cache READ, different question)...

Answer: Based on the documentation, prompt caching supports two TTL (Time To Live) durations:

1. **5-minute TTL** – The default cache duration supported by many models. The TTL resets with each successful ca...

⏱  Time:                3.29s
📥  Input tokens:        15
📤  Output tokens:       287
✍️  Cache WRITE tokens:  0
📖  Cache READ tokens:   1822  ← context served from cache


## 6. Third cached call — confirm cache is still warm

In [11]:
print("Running THIRD cached call (cache READ, another question)...")
cache_read2 = call_with_cache("Explain cache checkpoints.")

print(f"\nAnswer: {cache_read2['answer'][:200]}...")
print(f"\n⏱  Time:                {cache_read2['elapsed']:.2f}s")
print(f"📖  Cache READ tokens:   {cache_read2['cache_read_tokens']}  ← served from cache again")

Running THIRD cached call (cache READ, another question)...

Answer: # Cache Checkpoints Explained

## What Are Cache Checkpoints?

Cache checkpoints are **markers within your prompt** that define a contiguous subsection (called a "prompt prefix") that you want Amazon ...

⏱  Time:                7.01s
📖  Cache READ tokens:   1822  ← served from cache again


## 7. Results Comparison

In [7]:
print("=" * 60)
print("PROMPT CACHING — RESULTS SUMMARY")
print("=" * 60)

print(f"\n{'Metric':<30} {'No Cache':>12} {'Cache WRITE':>12} {'Cache READ':>12}")
print("-" * 68)
print(f"{'Latency (seconds)':<30} {baseline['elapsed']:>11.2f}s {cache_write['elapsed']:>11.2f}s {cache_read['elapsed']:>11.2f}s")
print(f"{'Input tokens':<30} {baseline['input_tokens']:>12} {cache_write['input_tokens']:>12} {cache_read['input_tokens']:>12}")
print(f"{'Cache write tokens':<30} {baseline['cache_write_tokens']:>12} {cache_write['cache_write_tokens']:>12} {cache_read['cache_write_tokens']:>12}")
print(f"{'Cache read tokens':<30} {baseline['cache_read_tokens']:>12} {cache_write['cache_read_tokens']:>12} {cache_read['cache_read_tokens']:>12}")

speedup = baseline['elapsed'] / cache_read['elapsed'] if cache_read['elapsed'] > 0 else 0
token_saving = baseline['input_tokens'] - cache_read['input_tokens']

print("\n" + "=" * 60)
print("KEY TAKEAWAYS")
print("=" * 60)
print(f"  Latency speedup on cache hit:  {speedup:.1f}x faster")
print(f"  Input tokens saved per call:   {token_saving:,}")
print(f"\n  Cache WRITE: slightly slower (Bedrock saves the prefix)")
print(f"  Cache READ:  faster + cheaper (prefix retrieved, not reprocessed)")
print(f"\n  Cache READ tokens are billed at ~10% of normal input token cost")
print(f"  The more you reuse the same prefix, the bigger the saving.")
print("=" * 60)

PROMPT CACHING — RESULTS SUMMARY

Metric                             No Cache  Cache WRITE   Cache READ
--------------------------------------------------------------------
Latency (seconds)                     6.95s        6.41s        3.17s
Input tokens                           1839           17           15
Cache write tokens                        0         1822            0
Cache read tokens                         0            0         1822

KEY TAKEAWAYS
  Latency speedup on cache hit:  2.2x faster
  Input tokens saved per call:   1,824

  Cache WRITE: slightly slower (Bedrock saves the prefix)
  Cache READ:  faster + cheaper (prefix retrieved, not reprocessed)

  Cache READ tokens are billed at ~10% of normal input token cost
  The more you reuse the same prefix, the bigger the saving.


## Key Concepts Summary

| Concept | Detail |
|---|---|
| **Cache point** | A `cachePoint` block in your system prompt or messages tells Bedrock what to cache |
| **Cache WRITE** | First call — Bedrock processes and stores the prefix. Slightly slower, small extra cost |
| **Cache READ** | Subsequent calls — Bedrock retrieves from cache. Faster, ~90% cheaper for cached tokens |
| **TTL** | Cache lives for ~5 minutes of inactivity. Each cache hit resets the timer |
| **Best for** | Large stable contexts reused across many queries: system prompts, documents, codebases |
